In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/adult-census-income/adult.csv


## Loading Dataset

In [2]:
df = pd.read_csv('/kaggle/input/adult-census-income/adult.csv')

## Exporting necessary Libraries

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  OrdinalEncoder,OneHotEncoder,LabelEncoder
from sklearn.preprocessing import MinMaxScaler

## Exploring Data

In [4]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [5]:
df.shape

(32561, 15)

# Handling Special Characters

## Identify special Characters

In [6]:
special_char = '\\?'
columns_with_special_char = df.apply(lambda col: col.astype(str).str.contains(special_char, regex=True).any(), axis=0)
special_char_columns = columns_with_special_char[columns_with_special_char].index.tolist()

## Replace witn nan

In [7]:
df = df.replace('?', np.nan)

# Handle Missing Values

## Identify Missing Values

In [8]:
df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64

## Identify numerical and categorical columns

In [9]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

## Imputing Missing Values

In [10]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].mean())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [11]:
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

# Dropping Uncessary Column

In [12]:
df = df.drop(columns=['fnlwgt'])

# Encoding Categorical Variables

## Identify Categorical Columns

In [13]:
cat_cols = df.select_dtypes(include=['object']).columns

print("Categorical columns:", cat_cols)

Categorical columns: Index(['workclass', 'education', 'marital.status', 'occupation',
       'relationship', 'race', 'sex', 'native.country', 'income'],
      dtype='object')


In [14]:
for col in cat_cols:
    unique_values = df[col].unique()
    print(f"Unique values in '{col}':", unique_values)

Unique values in 'workclass': ['Private' 'State-gov' 'Federal-gov' 'Self-emp-not-inc' 'Self-emp-inc'
 'Local-gov' 'Without-pay' 'Never-worked']
Unique values in 'education': ['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate' 'Prof-school'
 'Bachelors' 'Masters' '11th' 'Assoc-acdm' 'Assoc-voc' '1st-4th' '5th-6th'
 '12th' '9th' 'Preschool']
Unique values in 'marital.status': ['Widowed' 'Divorced' 'Separated' 'Never-married' 'Married-civ-spouse'
 'Married-spouse-absent' 'Married-AF-spouse']
Unique values in 'occupation': ['Prof-specialty' 'Exec-managerial' 'Machine-op-inspct' 'Other-service'
 'Adm-clerical' 'Craft-repair' 'Transport-moving' 'Handlers-cleaners'
 'Sales' 'Farming-fishing' 'Tech-support' 'Protective-serv' 'Armed-Forces'
 'Priv-house-serv']
Unique values in 'relationship': ['Not-in-family' 'Unmarried' 'Own-child' 'Other-relative' 'Husband' 'Wife']
Unique values in 'race': ['White' 'Black' 'Asian-Pac-Islander' 'Other' 'Amer-Indian-Eskimo']
Unique values in 'sex': ['Female'

## For Nominal Categories

In [15]:
selected_cat_cols = ['workclass', 'race']
print("Selected columns for encoding:", selected_cat_cols)

# Apply One-Hot Encoding
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
onehot_encoded = onehot_encoder.fit_transform(df[selected_cat_cols])

# Convert the one-hot encoded array to a DataFrame
onehot_encoded_df = pd.DataFrame(onehot_encoded, columns=onehot_encoder.get_feature_names_out(selected_cat_cols))

# Merge one-hot encoded DataFrame with the original DataFrame (excluding original categorical columns)
df = df.drop(selected_cat_cols, axis=1).join(onehot_encoded_df)

print("One-hot encoding complete for selected categorical columns.")
print(df.head())  # Display the first few rows of the updated DataFrame

Selected columns for encoding: ['workclass', 'race']
One-hot encoding complete for selected categorical columns.
   age     education  education.num marital.status         occupation  \
0   90       HS-grad              9        Widowed     Prof-specialty   
1   82       HS-grad              9        Widowed    Exec-managerial   
2   66  Some-college             10        Widowed     Prof-specialty   
3   54       7th-8th              4       Divorced  Machine-op-inspct   
4   41  Some-college             10      Separated     Prof-specialty   

    relationship     sex  capital.gain  capital.loss  hours.per.week  ...  \
0  Not-in-family  Female             0          4356              40  ...   
1  Not-in-family  Female             0          4356              18  ...   
2      Unmarried  Female             0          4356              40  ...   
3      Unmarried  Female             0          3900              40  ...   
4      Own-child  Female             0          3900           

## For Ordinal Categories

In [16]:
ordinal_categories = [['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th', 'HS-grad', 'Some-college', 'Assoc-acdm', 'Assoc-voc', 'Bachelors', 'Masters', 'Doctorate', 'Prof-school']]
ordinal_col = 'education'

ordinal_encoder = OrdinalEncoder(categories=ordinal_categories)
df[ordinal_col] = ordinal_encoder.fit_transform(df[[ordinal_col]])

df.head()

,age,education,education.num,marital.status,occupation,relationship,sex,capital.gain,capital.loss,hours.per.week,...,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass_Without-pay,race_Amer-Indian-Eskimo,race_Asian-Pac-Islander,race_Black,race_Other,race_White
0,90,8.0,9,Widowed,Prof-specialty,Not-in-family,Female,0,4356,40,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,82,8.0,9,Widowed,Exec-managerial,Not-in-family,Female,0,4356,18,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,66,9.0,10,Widowed,Prof-specialty,Unmarried,Female,0,4356,40,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,54,3.0,4,Divorced,Machine-op-inspct,Unmarried,Female,0,3900,40,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,41,9.0,10,Separated,Prof-specialty,Own-child,Female,0,3900,40,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## Label Encoding

In [17]:
label_encoder = LabelEncoder()
df['income'] = label_encoder.fit_transform(df['income'])

# Feature Scaling

## Min-Max Scaling

In [18]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

min_max_scaler = MinMaxScaler()
df[num_cols] = min_max_scaler.fit_transform(df[num_cols])

print(df.head())

        age  education  education.num marital.status         occupation  \
0  1.000000   0.533333       0.533333        Widowed     Prof-specialty   
1  0.890411   0.533333       0.533333        Widowed    Exec-managerial   
2  0.671233   0.600000       0.600000        Widowed     Prof-specialty   
3  0.506849   0.200000       0.200000       Divorced  Machine-op-inspct   
4  0.328767   0.600000       0.600000      Separated     Prof-specialty   

    relationship     sex  capital.gain  capital.loss  hours.per.week  ...  \
0  Not-in-family  Female           0.0      1.000000        0.397959  ...   
1  Not-in-family  Female           0.0      1.000000        0.173469  ...   
2      Unmarried  Female           0.0      1.000000        0.397959  ...   
3      Unmarried  Female           0.0      0.895317        0.397959  ...   
4      Own-child  Female           0.0      0.895317        0.397959  ...   

  workclass_Private  workclass_Self-emp-inc  workclass_Self-emp-not-inc  \
0          

# Splitting the Dataset

In [19]:
X = df.drop('income', axis=1) 
y = df['income']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the shapes of the resulting datasets
print(f'Training set shape: X={X_train.shape}, y={y_train.shape}')
print(f'Testing set shape: X={X_test.shape}, y={y_test.shape}')

Training set shape: X=(26048, 24), y=(26048,)
Testing set shape: X=(6513, 24), y=(6513,)
